In [8]:
import numpy as np
import pandas as pd
import patsy
from tensorzinb.tensorzinb import TensorZINB


2025-03-27 16:53:29.450226: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-27 16:53:29.454577: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [3]:
fake_cres=pd.read_csv("fake_cres.csv").drop("Unnamed: 0",axis=1)
fake_cres

,CRE,Cell_type,replicate_ID,umi_count
0,nobody,brain,1,0
1,nobody,brain,1,0
2,nobody,brain,1,0
3,nobody,brain,1,0
4,nobody,brain,1,0
...,...,...,...,...
14307,neurogene,blood,3,7
14308,neurogene,blood,3,26
14309,neurogene,blood,3,7
14310,neurogene,blood,3,15


In [4]:
fake_cres_munged=fake_cres
fake_cres_munged["replicate_ID"]=fake_cres_munged["replicate_ID"].map({1:"rep1",2:"rep2",3:"rep3"})

In [5]:
y, X = patsy.dmatrices("umi_count ~ C(CRE)*C(Cell_type)-1",
                        fake_cres_munged, return_type='dataframe')
Z = patsy.dmatrix("C(replicate_ID)", fake_cres_munged, return_type='dataframe')

In [15]:
y

,umi_count
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
14307,7.0
14308,26.0
14309,7.0
14310,15.0


array([[ 0.],
       [ 0.],
       [ 0.],
       ...,
       [ 7.],
       [15.],
       [38.]])

In [54]:
zinbo=TensorZINB(y["umi_count"].to_numpy().reshape((-1,1)),X,exog_infl=Z.to_numpy())#,same_dispersion=True
zinb_result=zinbo.fit(init_method="nb")

In [55]:
zinb_result

{'llf_total': -21228.66048024269,
 'llfs': array([-21228.66048024]),
 'aic_total': 42485.32096048538,
 'aics': array([42485.32096049]),
 'df_model_total': 14,
 'df': 14,
 'weights': {'x_mu': array([[ 4.6394043 ],
         [ 2.7402766 ],
         [ 0.66553664],
         [ 4.658981  ],
         [ 2.5248148 ],
         [ 0.03926665],
         [ 1.7864101 ],
         [-0.67963153],
         [-1.2742932 ],
         [-0.26394197]], dtype=float32),
  'x_pi': array([[ 1.3557478],
         [-1.3266021],
         [ 0.8895673]], dtype=float32),
  'theta': array([[1.249529]], dtype=float32)},
 'cpu_time': 2.3001821041107178,
 'num_sample': 14312,
 'epochs': 666}

# Recapitulating dispersion constant

In [59]:
np.exp(zinb_result['weights']['theta'][0])

array([3.4886992], dtype=float32)

If I am correct in exponentiating it (I think I am), that's pretty close to the ground truth value of 3.333

# Recapitulating zero-inflation parameters

In [60]:
#I think x_pi are the weights on 
zinb_result['weights']['x_pi']

array([[ 1.3557478],
       [-1.3266021],
       [ 0.8895673]], dtype=float32)

In [65]:
dropout_design_matrix=Z.drop_duplicates().to_numpy()

In [69]:
dropout_design_matrix

array([[1., 0., 0.],
       [1., 1., 0.],
       [1., 0., 1.]])

In [70]:
pis=dropout_design_matrix.dot(zinb_result['weights']['x_pi'])
pis

array([[1.35574782],
       [0.02914572],
       [2.24531513]])

In [75]:
#hrm. I bet these are bernouli constants passed through logit. 
#Let's undo w/ logistic function
1/(1+np.exp(-pis))

array([[0.79506774],
       [0.50728591],
       [0.90424566]])

If we assume the order is rep1, rep2, rep3, then the real values are 0.8, 0.5, 0.9.

That's pretty damn close!

# Recapitulating $\mu$s (mean parameters).

In [76]:
#we begin by getting all combinations of the predictors (which are of course all categorical) present in the data. 
minimal_nb_design = X.drop_duplicates()
minimal_nb_design

,C(CRE)[everybody],C(CRE)[neurogene],C(CRE)[nobody],C(CRE)[redgene],C(CRE)[somebody],C(Cell_type)[T.brain],C(CRE)[T.neurogene]:C(Cell_type)[T.brain],C(CRE)[T.nobody]:C(Cell_type)[T.brain],C(CRE)[T.redgene]:C(Cell_type)[T.brain],C(CRE)[T.somebody]:C(Cell_type)[T.brain]
0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
440,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
904,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1355,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1798,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2263,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2762,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3266,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3767,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4262,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [97]:
#examining the table above, we reconstruct the index
recapitulated_nb_rate=pd.DataFrame({"cre":["nobody","somebody","everybody","redgene","neurogene","nobody","somebody","everybody","redgene","neurogene"],
    "cell_type":["brain"]*5+["blood"]*5})
recapitulated_nb_rate

,cre,cell_type
0,nobody,brain
1,somebody,brain
2,everybody,brain
3,redgene,brain
4,neurogene,brain
5,nobody,blood
6,somebody,blood
7,everybody,blood
8,redgene,blood
9,neurogene,blood


In [79]:
zinb_result['weights']['x_mu']

array([[ 4.6394043 ],
       [ 2.7402766 ],
       [ 0.66553664],
       [ 4.658981  ],
       [ 2.5248148 ],
       [ 0.03926665],
       [ 1.7864101 ],
       [-0.67963153],
       [-1.2742932 ],
       [-0.26394197]], dtype=float32)

In [80]:
len(zinb_result['weights']['x_mu'])

10

That's the same as the number of columns in our design matrix. Assuming orientation was preserved...

In [94]:
np.exp(minimal_nb_design.dot(zinb_result['weights']['x_mu']))

,0
0,1.025491
440,9.975574
904,107.626936
1355,30.690535
1798,96.154216
2263,1.945534
2762,12.488583
3266,103.482684
3767,105.528478
4262,15.491269


In [98]:
recapitulated_nb_rate["expected_value"]=np.exp(minimal_nb_design.dot(zinb_result['weights']['x_mu'])).to_numpy()
#exponent to undo log link function
recapitulated_nb_rate

,cre,cell_type,expected_value
0,nobody,brain,1.025491
1,somebody,brain,9.975574
2,everybody,brain,107.626936
3,redgene,brain,30.690535
4,neurogene,brain,96.154216
5,nobody,blood,1.945534
6,somebody,blood,12.488583
7,everybody,blood,103.482684
8,redgene,blood,105.528478
9,neurogene,blood,15.491269


Comparing to the ground-truth:

In [99]:
REAL_data = {
    "CRE": ["nobody", "somebody", "everybody", "redgene", "neurogene", "nobody", "somebody", "everybody", "redgene", "neurogene"],
    "Cell-type": ["brain", "brain", "brain", "brain", "brain", "blood", "blood", "blood", "blood", "blood"],
    "mean": [1, 10, 114, 30, 99, 2, 12, 109, 112, 16]
}

# Creating the dataframe
pd.DataFrame(REAL_data)

,CRE,Cell-type,mean
0,nobody,brain,1
1,somebody,brain,10
2,everybody,brain,114
3,redgene,brain,30
4,neurogene,brain,99
5,nobody,blood,2
6,somebody,blood,12
7,everybody,blood,109
8,redgene,blood,112
9,neurogene,blood,16


Pretty good !